# v20 — All Optuna Best Params (LGBM + ET + CatBoost)

Final combined experiment applying all Optuna-tuned params to the full stacking pipeline.

**Changes vs v16 baseline ($21,552 OOF):**
- LGBM: Optuna 100-trial TPE best — nb 13 (individual val $21,518 vs $21,776)
- ET: Optuna 50-trial best — nb 13 (600 trees, depth 24 vs 300 trees, depth 20)
- CatBoost: Optuna 100-trial TPE best — nb 20 (individual val $21,484 vs $22,607, −$1,123!)
- XGB: unchanged (Optuna marginal individual gain, stack risk not worth it)

Only edit Cell 1 (CONFIG). Run all cells. Compare Cell 3 OOF RMSE vs v19 ($21,533).

In [ ]:
# ── EDIT ONLY THIS CELL ────────────────────────────────────────────────────
CONFIG = {
    "experiment_name": "v20",  # Change per experiment — used as MLflow run label

    "data": {
        "train_path": "../../data/raw/train.csv",
        "test_path":  "../../data/raw/test.csv",
    },

    "features": {
        # OOF target encodings — same as v16 baseline
        "encode_pairs": [
            ("town",           "town_median_price",           1),
            ("postal_sector",  "postal_sector_median_price",  1),
            ("flat_model",     "flat_model_median_price",     1),
            ("town_flat_type", "town_flat_type_median_price", 3),
            ("Tranc_Year",     "year_median_price",           1),
        ],
        "drop_cols": [],
    },

    "models": {
        "active": ["xgb", "lgbm", "et", "catboost"],

        # XGB — unchanged from v16 (Optuna individual gain marginal; stack already balanced)
        "xgb": {
            "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.4, "min_child_weight": 7,
            "reg_alpha": 0.01, "reg_lambda": 1.5,
            "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
        },

        # LGBM — Optuna TPE 100-trial best (nb 13): individual val $21,518 vs $21,776
        "lgbm": {
            "n_estimators": 888, "max_depth": 14, "num_leaves": 317,
            "learning_rate": 0.0272, "subsample": 0.9225, "colsample_bytree": 0.4564,
            "min_child_samples": 31, "reg_alpha": 0, "reg_lambda": 1.135,
            "random_state": 42, "n_jobs": -1, "verbosity": -1,
        },

        # ET — Optuna 50-trial best (nb 13): 600 trees, depth 24
        "et": {
            "n_estimators": 600, "min_samples_split": 7, "min_samples_leaf": 2,
            "max_features": 0.829, "max_depth": 24,
            "random_state": 42, "n_jobs": -1,
        },

        # CatBoost — Optuna TPE 100-trial best (nb 20): individual val $21,484 vs $22,607 (−$1,123!)
        "catboost": {
            "iterations": 1994, "depth": 10,
            "learning_rate": 0.0585,
            "l2_leaf_reg": 1.4962, "colsample_bylevel": 0.7696,
            "min_child_samples": 20,
            "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
        },

        "ridge_alphas": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    },

    "cv": {"n_splits": 5, "random_state": 42},

    "output": {
        "submission_path": "../../outputs/submissions/sub_v20.csv",
        # "fulldata_submission_path": "../../outputs/submissions/sub_v20_fulldata.csv",
        "sample_path": "../../outputs/submissions/sample_sub_reg.csv",
    },
}

In [ ]:
import sys
sys.path.insert(0, "../../")

from src.pipeline import run_pipeline

results = run_pipeline(CONFIG)

In [ ]:
print(f'OOF RMSE:      ${results["oof_rmse"]:,.0f}')
print(f'Ridge weights: {results["weights"]}')
print(f'Best alpha:    {results["best_alpha"]}')
print()
print('Per-model mean OOF RMSE:')
import numpy as np
for m, scores in results['fold_scores'].items():
    print(f'  {m:<12} ${np.mean(scores):>10,.0f}')
print()
print('To view all experiments in MLflow UI:')
print('  cd <project_root>')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('  Open http://localhost:5000')